# Module B1 & B2: Text Preprocessing and Sentiment Analysis

This notebook demonstrates text cleaning (preprocessing, stopword removal, and lemmatization) using NLTK on product review data, and trains a TF-IDF + Logistic Regression model to classify sentiment into positive, negative, or neutral categories.

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Ensure NLTK resources are downloaded
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print('Libraries imported.')

## 1. Load Dataset

In [ ]:
reviews_path = '../data/reviews.csv'
df = pd.read_csv(reviews_path)
print(df.head())
print('\nSentiment counts:\n', df['sentiment'].value_counts())

## 2. Text Preprocessing Pipeline
We will implement:
- Lowercasing
- Punctuation removal
- Tokenization
- Stopword removal
- Lemmatization

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    # Lowercase
    text = text.lower()
    # Punctuation removal
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Tokenization
    tokens = word_tokenize(text)
    # Stopword removal and lemmatization
    cleaned = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return ' '.join(cleaned)

df['cleaned_text'] = df['review_text'].apply(preprocess_text)
print('Sample original text:', df['review_text'].iloc[0])
print('Sample cleaned text:', df['cleaned_text'].iloc[0])

## 3. Vectorization (TF-IDF)
Convert preprocessed text to numerical features using TF-IDF.

In [ ]:
X = df['cleaned_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f'Vocabulary size: {len(vectorizer.vocabulary_)}')
print('X_train_vec shape:', X_train_vec.shape)

## 4. Train Classifier (Logistic Regression)

In [ ]:
model = LogisticRegression(class_weight='balanced')
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred))

## 5. Save Models
We save both the trained TF-IDF Vectorizer and the Logistic Regression model.

In [ ]:
os.makedirs('../app/models', exist_ok=True)
with open('../app/models/sentiment_model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('../app/models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print('Sentiment model and vectorizer saved successfully.')